# 分類系統的設計

dr-chenglung: 以下分類類別的定義，準確度想要再提升，請建議模型可以怎麼設計?如何提取與利用base_model更多的資訊，去提升classifier分類的準確性。請先構想給我幾個方案，之後再提供範例程式碼。

GitHub Copilot: # 提升新聞分類器準確度的方案

分析您現有的模型架構，我發現幾個可以進一步提升分類準確度的方向。目前模型使用 Qwen2.5-0.5B-instruct 作為基礎，僅平均所有 token 的 hidden states，並通過多層分類器進行預測。以下是幾個可以提升準確度的策略：

## 方案一：改進特徵提取方式

現有模型僅使用平均池化獲取句子表示，可以改進為：

1. **注意力池化機制**：讓模型學習哪些 tokens 對分類更重要
2. **多視角特徵提取**：結合平均池化、最大池化等多種表示方法
3. **自注意力機制**：基於內容自適應地為不同詞賦予權重

## 方案二：多層表示融合

目前只使用最後一層隱藏狀態，可以考慮：

1. **多層融合**：結合模型不同層的表示（如最後4層）
2. **層間注意力**：用注意力機制融合不同層的輸出
3. **層間殘差連接**：使用殘差連接更有效地融合不同層的特徵


## 方案三：模型參數選擇性微調

1. **LoRA微調**：只微調部分關鍵參數矩陣
2. **選擇性解凍**：解凍最後幾層Transformer層進行微調
3. **Adapter模塊**：為基礎模型添加小型的特定任務適配器

# Transformers變形金剛(變壓器)模型

<img src="https://cdn.analyticsvidhya.com/wp-content/uploads/2019/06/Screenshot-from-2019-06-17-19-53-10.png">

Bert and GPT are parts of Transformer

<img src="https://heidloff.net/assets/img/2023/02/transformers.png" width="800">

Encoders and Decoders
As mentioned, there are encoders and decoders. BERT uses encoders only, GTP uses decoders only. Both options understand language including syntax and semantics. Especially the next generation of large language models like GPT with billions of parameters do this very well.

The two models focus on different scenarios. However, since the field of foundation models is evolving, the differentiation is often fuzzier.

BERT (encoder): classification (e.g., sentiment), questions and answers, summarization, named entity recognition
GPT (decoder): translation, generation (e.g., stories)
The outputs of the core models are different:

BERT (encoder): Embeddings representing words with attention information in a certain context
GPT (decoder): Next words with probabilities
Both models are pretrained and can be reused without intensive training. Some of them are available as open source and can be downloaded from communities like Hugging Face, others are commercial. Reuse is important, since trainings are often very resource intensive and expensive which few companies can afford.

The pretrained models can be extended and customized for different domains and specific tasks. Layers can sometimes be reused without modifications and more layers are added on top. If layers need to be modified, the new training is more expensive. The technique to customize these models is called Transfer Learning, since the same generic model can easily be transferred to other domains.

[Source](https://heidloff.net/article/foundation-models-transformers-bert-and-gpt/)

### Encoder-Decoder Transformers



<img src='https://miro.medium.com/v2/resize:fit:1400/format:webp/1*vrSX_Ku3EmGPyqF_E-2_Vg.png'>

### Attension Layer
<img src='https://miro.medium.com/v2/resize:fit:640/format:webp/1*aTLu4HxVUzEOIZTLHmOktw.png'>

# Insatll packages

In [1]:
# !pip install evaluate

In [2]:
import torch
import datasets
import pandas as pd
import evaluate
import numpy as np

# Load Huggingface transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sklearn import metrics
import torch
import evaluate
import datasets


In [3]:
# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


# Load Tokenizer

In [4]:
model_id = "Qwen/Qwen2.5-0.5B-instruct"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id)


#  定義Model

這裡的做法有簡單的版本也有複雜的版本

簡單版:


        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,  # 模型名稱
            num_labels=len(categories),  # Number of output labels
        )


複雜版:




In [6]:
from transformers import Qwen2Model, Trainer, TrainingArguments, AutoTokenizer, AutoModel
from torch import nn
from transformers.modeling_outputs import SequenceClassifierOutput
import os

In [7]:
import torch
import os
import torch.nn.functional as F
from torch import nn

class QwenForClassifier(nn.Module):
    def __init__(self, base_model, hidden_size, num_labels, num_fusion_layers=4):
        super(QwenForClassifier, self).__init__()
        # 指定基礎模型 來自於外部指定的模型
        self.base_model = base_model
        
        # 融合權重多少層 (可調整層數)
        self.num_fusion_layers = num_fusion_layers
        # 保存config配置
        self.config = base_model.config
        self.config.num_labels = num_labels
        
        # 凍結 base model 的參數
        for param in self.base_model.parameters():
            param.requires_grad = False
        
        # 多層融合權重 (最後4層)
        # 初始值是0.25，訓練後模型可能會學到最後一層權重為0.4，倒數第二層為0.3，倒數第三層為0.2，倒數第四層為0.1
        self.layer_weights = nn.Parameter(torch.ones(num_fusion_layers) / num_fusion_layers)
            
        # 新增 QKV 線性層
        self.q_proj = nn.Linear(hidden_size, hidden_size) # size (896, 896)
        self.k_proj = nn.Linear(hidden_size, hidden_size) # size (896, 896)
        self.v_proj = nn.Linear(hidden_size, hidden_size) # size (896, 896)

        # 增強型分類器
        # 這是一個三層的神經網絡，將高維特徵映射到類別空間：
        #     1. 第一層：將1024維降到256維
        #     2. 第二層：將256維降到128維
        #     3. 第三層：將128維降到類別數量
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            
            nn.Linear(128, num_labels)
        )
        
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        # 獲取所有隱藏層狀態
        outputs = self.base_model(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            output_hidden_states=True # 設置為True以獲取所有隱藏層狀態
        )
        
        # outputs 是一個包含多個元素的元組
        hidden_states = outputs.hidden_states # 有24層的隱藏狀態 維度是
        # 獲取最後4層隱藏狀態
        
        last_layers = hidden_states[-self.num_fusion_layers:] # 取得最後4層的隱藏狀態 維度是 (batch_size, seq_len, hidden_size) 
        layer_weights = F.softmax(self.layer_weights, dim=0) # 將權重轉換為概率分佈 維度是 (num_fusion_layers,)
        # 加權融合最後4層特徵
        sequence_output = torch.zeros_like(last_layers[0]) # 初始化為0，維度是 (batch_size, seq_len, hidden_size)
        for i, layer in enumerate(last_layers):
            sequence_output += layer_weights[i].unsqueeze(-1).unsqueeze(-1) * layer # 將權重擴展到 (batch_size, seq_len, hidden_size)
    
        # ===== QKV Attention Pooling =====
        # [batch, seq_len, hidden_size]
        Q = self.q_proj(sequence_output)
        K = self.k_proj(sequence_output)
        V = self.v_proj(sequence_output)
        # Attention scores: [batch, seq_len, seq_len]
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (Q.size(-1) ** 0.5)
        attn_probs = F.softmax(attn_scores, dim=-1)
        # Context vector: [batch, seq_len, hidden_size]
        context_vector_qkv = torch.matmul(attn_probs, V).mean(dim=1)  # [batch, hidden_size]

        # ===== mean pooling =====
        mean_pooled = torch.mean(sequence_output, dim=1)

        # ===== 融合 QKV context 與 mean_pooled =====
        combined_repr = context_vector_qkv + mean_pooled

        # 分類預測
        logits = self.classifier(combined_repr) # 維度是 (batch_size, num_labels)
        
        # 計算損失
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss() # 計算交叉熵損失
            loss = loss_fct(logits, labels)
            
        return {"loss": loss, "logits": logits}
    
    def save_model(self, output_dir=None):
        """保存分類器權重和配置"""
        os.makedirs(output_dir, exist_ok=True)
        
        # 保存分類器權重與QKV權重
        classifier_path = os.path.join(output_dir, "classifier_weights.pt")
        model_dict = {
            'classifier': self.classifier.state_dict(),
            'layer_weights': self.layer_weights,
            'q_proj': self.q_proj.state_dict(),
            'k_proj': self.k_proj.state_dict(),
            'v_proj': self.v_proj.state_dict(),
            'config': {
                'num_labels': self.config.num_labels,
                'hidden_size': self.config.hidden_size
            }
        }
        torch.save(model_dict, classifier_path)
        print(f"已保存分類器權重至 {classifier_path}")
    
    def load_model(self, model_dir, device=None):
        """載入分類器權重"""
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            
        classifier_path = os.path.join(model_dir, "classifier_weights.pt")
        if os.path.exists(classifier_path):
            model_dict = torch.load(classifier_path, map_location=device, weights_only=True)
            
            # 載入各組件
            self.classifier.load_state_dict(model_dict['classifier'])
            self.layer_weights.data = model_dict['layer_weights'].to(device)
            self.q_proj.load_state_dict(model_dict['q_proj'])
            self.k_proj.load_state_dict(model_dict['k_proj'])
            self.v_proj.load_state_dict(model_dict['v_proj'])
            
            print(f"已載入分類器權重: {classifier_path}")
            return True
        else:
            print(f"警告: 找不到分類器權重檔案 {classifier_path}")
            return False

## 初始化模型


        分類任務：兩者都可用，但 AutoModel 更輕量
        生成功能：只有 AutoModelForCausalLM 支持
        內存使用：AutoModelForCausalLM 通常較大，因為包含了完整的語言模型頭
        在 Qwen2 情感分類模型中，使用 AutoModelForCausalLM 更為靈活，因為它既可以進行分類，也保留了原始的文本生成功能。



        # AutoModelForCausalLM 輸出
        outputs = base_model(input_ids, attention_mask)
        # 輸出包含：
        # - loss: (可選) 語言模型損失
        # - logits: 張量，形狀為 [batch_size, sequence_length, vocab_size]
        # - past_key_values: (可選) 用於加速解碼的過去狀態
        # - hidden_states: (可選) 所有隱藏層狀態的元組
        # - attentions: (可選) 注意力權重的元組


        # AutoModel 輸出
        outputs = base_model(input_ids, attention_mask)
        # 輸出包含：
        # - last_hidden_state: 張量，形狀為 [batch_size, sequence_length, hidden_size]
        # - hidden_states: (可選) 所有隱藏層狀態的元組
        # - attentions: (可選) 注意力權重的元組

In [8]:

categories=['負面','正面']
len(categories)

2

In [9]:
# 在外部先載入base_model預訓練權重(不包含分類層)
full_model = AutoModelForCausalLM.from_pretrained(model_id)
#
hidden_size = full_model.config.hidden_size
model = QwenForClassifier(full_model.model, hidden_size, num_labels= len(categories))

# 移動到指定設備
model = model.to(device)

In [10]:
full_model.model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

In [11]:
full_model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

## 看看base_model與model有何不同?

model內部有: 一個base_model+輸出分類層。它被拼接為分類器，可以做兩個類別的分類

base_model仍舊是GPT自回歸模型

但是在記憶體，兩者是共享同一份base_model權重，model有多一個分類器的輸出層的部分

In [12]:
full_model 

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [13]:
model

QwenForClassifier(
  (base_model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMS

In [14]:
# 這與model是一樣的模型??
model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

## 基本分解展示

In [15]:
text = "我很開心"
# Tokenize the input text
inputs = tokenizer(
    text,
    max_length=512,
    truncation=True,
    return_tensors="pt"
).to(device)
inputs

{'input_ids': tensor([[106922,  86347,  63109]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1]], device='cuda:0')}

In [16]:
tokenizer.decode([106922])

'我很'

In [17]:
tokenizer.decode([86347])

'開'

In [18]:
tokenizer.decode([63109])

'心'

In [19]:
# Get model predictions
with torch.no_grad():
    outputs = full_model.model(**inputs, output_hidden_states=True)
outputs

BaseModelOutputWithPast(last_hidden_state=tensor([[[ 0.9264, -2.3442,  7.2255,  ...,  1.5260, -0.6406,  8.6983],
         [ 8.1989,  5.0171,  5.4528,  ...,  2.3196,  0.1038, 17.0922],
         [ 4.1365,  2.7577, -1.9487,  ...,  4.2585,  0.3259,  5.6416]]],
       device='cuda:0'), past_key_values=((tensor([[[[-8.3076e+00, -3.1791e+00, -6.2214e+00,  9.5528e-01, -1.6035e-01,
            9.2531e+00,  8.4073e+00, -1.5394e+00,  1.4039e-01, -1.9616e-01,
           -1.7907e-01,  2.5515e-02, -2.4031e+01,  7.8197e-02,  2.0908e+01,
           -7.1390e-01, -1.4523e-01, -2.3799e+00,  1.8700e+01, -2.1435e+01,
            8.1560e+00, -5.7038e+00, -1.1518e+01,  1.6371e+01, -3.1596e+01,
            3.6830e+00,  3.3879e+01,  5.4809e+00,  3.2926e+01,  1.1170e+02,
            3.6837e+01,  1.2074e+02,  6.5178e+00,  8.8754e+00,  3.8099e+00,
           -6.5642e+00, -4.6886e-02, -1.2308e+00, -1.2553e+00,  9.0623e+00,
            1.6282e-01, -4.5582e-02,  5.6282e-03, -9.1971e-02,  9.9192e+00,
           -5.82

In [20]:
print(outputs.last_hidden_state.shape)
outputs.last_hidden_state

torch.Size([1, 3, 896])


tensor([[[ 0.9264, -2.3442,  7.2255,  ...,  1.5260, -0.6406,  8.6983],
         [ 8.1989,  5.0171,  5.4528,  ...,  2.3196,  0.1038, 17.0922],
         [ 4.1365,  2.7577, -1.9487,  ...,  4.2585,  0.3259,  5.6416]]],
       device='cuda:0')

In [21]:
hidden_states = outputs.hidden_states
print(len(hidden_states))
hidden_states

25


(tensor([[[-0.0065, -0.0226,  0.0020,  ...,  0.0098, -0.0023,  0.0001],
          [ 0.0056, -0.0129,  0.0031,  ...,  0.0133,  0.0254,  0.0139],
          [ 0.0035,  0.0030,  0.0146,  ..., -0.0056,  0.0047,  0.0215]]],
        device='cuda:0'),
 tensor([[[ 0.2450,  0.1225,  0.1153,  ...,  0.5541,  0.2049,  0.1885],
          [ 0.4195,  0.1024, -0.0947,  ...,  0.0015,  0.0347, -0.1258],
          [ 0.3389, -0.0468, -0.1289,  ...,  0.0539, -0.0965, -0.3557]]],
        device='cuda:0'),
 tensor([[[ 0.2829,  0.2549,  0.2188,  ...,  0.3216,  0.3285, -0.2504],
          [ 0.6255,  0.0886,  0.1458,  ..., -0.0729,  0.0526, -0.3206],
          [ 0.4053, -0.0124, -0.2143,  ...,  0.2401, -0.2040, -0.1618]]],
        device='cuda:0'),
 tensor([[[-0.3877, -6.5412, -2.1226,  ..., -0.9880, -0.4663,  3.2517],
          [ 0.6490,  0.1017, -0.0641,  ...,  0.0173,  0.1336, -0.2160],
          [ 0.6648, -0.1470, -0.0638,  ...,  0.3017, -0.3547, -0.1261]]],
        device='cuda:0'),
 tensor([[[-1.2781, -9.9

In [22]:
last_layers = hidden_states[-4:] # 取得最後4層的隱藏狀態 維度是 
last_layers

(tensor([[[-1.4841e+00, -1.0415e+01, -4.3223e+00,  ...,  1.1940e+00,
           -5.6780e-02,  8.0116e-02],
          [ 3.1205e+00,  1.6070e+00,  2.1115e-01,  ...,  7.6592e-01,
           -8.2045e-03, -1.2017e-01],
          [ 1.4784e+00,  1.4490e+00,  2.4957e-01,  ...,  1.8354e-01,
            4.8481e-01,  1.0272e+00]]], device='cuda:0'),
 tensor([[[ 0.5549, -1.5447,  0.2227,  ...,  0.2267,  0.2410,  0.4332],
          [ 3.6110,  1.4864,  2.2408,  ...,  0.9572, -0.7222, -0.2458],
          [ 1.6206,  1.7433, -0.0183,  ...,  0.2484,  0.3368,  0.0357]]],
        device='cuda:0'),
 tensor([[[ 0.5102,  0.0918,  1.1747,  ...,  0.3536, -1.1689,  2.2013],
          [ 3.6633,  2.0289,  2.3842,  ..., -1.3631,  0.3329, -2.1658],
          [ 1.4582,  1.9609, -0.1493,  ...,  0.2035,  0.3352, -1.0328]]],
        device='cuda:0'),
 tensor([[[ 0.9264, -2.3442,  7.2255,  ...,  1.5260, -0.6406,  8.6983],
          [ 8.1989,  5.0171,  5.4528,  ...,  2.3196,  0.1038, 17.0922],
          [ 4.1365,  2.7577

In [23]:

# 展示每一層的形狀
print("每一層的形狀:")
for i, layer in enumerate(last_layers):
    print(f"Layer -{4-i}: {layer.shape}")

每一層的形狀:
Layer -4: torch.Size([1, 3, 896])
Layer -3: torch.Size([1, 3, 896])
Layer -2: torch.Size([1, 3, 896])
Layer -1: torch.Size([1, 3, 896])


In [24]:
# 展示層權重
layer_weights = F.softmax(model.layer_weights, dim=0)
print("\n層權重:")
for i, weight in enumerate(layer_weights):
    print(f"Layer -{4-i} weight: {weight.item():.4f}")
    print( layer_weights[i])
    print( layer_weights[i].shape)
    print( layer_weights[i].unsqueeze(-1).unsqueeze(-1) ) # 連續兩次unsqueeze增加維度，成為2D張量
    print( layer_weights[i].unsqueeze(-1).unsqueeze(-1).shape )
    print()


層權重:
Layer -4 weight: 0.2500
tensor(0.2500, device='cuda:0', grad_fn=<SelectBackward0>)
torch.Size([])
tensor([[0.2500]], device='cuda:0', grad_fn=<UnsqueezeBackward0>)
torch.Size([1, 1])

Layer -3 weight: 0.2500
tensor(0.2500, device='cuda:0', grad_fn=<SelectBackward0>)
torch.Size([])
tensor([[0.2500]], device='cuda:0', grad_fn=<UnsqueezeBackward0>)
torch.Size([1, 1])

Layer -2 weight: 0.2500
tensor(0.2500, device='cuda:0', grad_fn=<SelectBackward0>)
torch.Size([])
tensor([[0.2500]], device='cuda:0', grad_fn=<UnsqueezeBackward0>)
torch.Size([1, 1])

Layer -1 weight: 0.2500
tensor(0.2500, device='cuda:0', grad_fn=<SelectBackward0>)
torch.Size([])
tensor([[0.2500]], device='cuda:0', grad_fn=<UnsqueezeBackward0>)
torch.Size([1, 1])



In [25]:
# 展示融合後的形狀
sequence_output = torch.zeros_like(last_layers[0])
print(f"\n融合後的輸出形狀: {sequence_output.shape}")
sequence_output


融合後的輸出形狀: torch.Size([1, 3, 896])


tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]], device='cuda:0')

In [26]:
# 加權融合最後4層特徵
for i, layer in enumerate(last_layers):
    sequence_output += layer_weights[i].unsqueeze(-1).unsqueeze(-1) * layer
print(f"\n加權融合後的輸出形狀: {sequence_output.shape}")
sequence_output


加權融合後的輸出形狀: torch.Size([1, 3, 896])


tensor([[[ 0.1269, -3.5531,  1.0752,  ...,  0.8251, -0.4063,  2.8532],
         [ 4.6484,  2.5348,  2.5722,  ...,  0.6699, -0.0734,  3.6401],
         [ 2.1734,  1.9777, -0.4667,  ...,  1.2235,  0.3707,  1.4179]]],
       device='cuda:0', grad_fn=<AddBackward0>)

In [27]:
# 加權融合最後4層特徵
sequence_output = torch.zeros_like(last_layers[0])
for i, layer in enumerate(last_layers):
    sequence_output += layer_weights[i] * layer
print(f"\n加權融合後的輸出形狀: {sequence_output.shape}")
sequence_output


加權融合後的輸出形狀: torch.Size([1, 3, 896])


tensor([[[ 0.1269, -3.5531,  1.0752,  ...,  0.8251, -0.4063,  2.8532],
         [ 4.6484,  2.5348,  2.5722,  ...,  0.6699, -0.0734,  3.6401],
         [ 2.1734,  1.9777, -0.4667,  ...,  1.2235,  0.3707,  1.4179]]],
       device='cuda:0', grad_fn=<AddBackward0>)

In [28]:
'''
所有情況的計算結果完全相同：無論使用幾次 unsqueeze，最終結果都是每個元素乘以相同的權重值。這是因為 PyTorch 的廣播規則會自動擴展維度較少的張量使其匹配維度較多的張量。

在模型中使用 unsqueeze(-1).unsqueeze(-1) 的理由：

明確性：使程式碼意圖更加清晰
通用性：確保與不同維度的輸入兼容
穩健性：防止當張量形狀變化時出現意外錯誤
在實際開發中，使用正確數量的 unsqueeze 可以確保程式碼的穩定性，即使當輸入張量形狀變化
'''
import torch

# 創建示例張量
weight = torch.tensor(0.25)  # 一個標量權重
feature = torch.tensor([[[1, 2, 3], 
                         [4, 5, 6]]])  # 形狀是 [1, 2, 3] 的張量

print("原始權重形狀:", weight.shape)
print("特徵張量形狀:", feature.shape)
print("原始權重值:", weight)
print("原始特徵值:\n", feature)
print("-" * 50)

# 0次 unsqueeze - 直接相乘
result0 = weight * feature
print("0次 unsqueeze 後權重形狀:", weight.shape)
print("相乘結果形狀:", result0.shape)
print("結果:\n", result0)
print("-" * 50)

# 1次 unsqueeze
weight1 = weight.unsqueeze(-1)
print("1次 unsqueeze 後權重形狀:", weight1.shape)
result1 = weight1 * feature
print("相乘結果形狀:", result1.shape)
print("結果:\n", result1)
print("-" * 50)

# 2次 unsqueeze
weight2 = weight.unsqueeze(-1).unsqueeze(-1)
print("2次 unsqueeze 後權重形狀:", weight2.shape)
result2 = weight2 * feature
print("相乘結果形狀:", result2.shape)
print("結果:\n", result2)
print("-" * 50)

# 3次 unsqueeze
weight3 = weight.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
print("3次 unsqueeze 後權重形狀:", weight3.shape)
result3 = weight3 * feature
print("相乘結果形狀:", result3.shape)
print("結果:\n", result3)
print("-" * 50)

# 4次 unsqueeze
weight4 = weight.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
print("4次 unsqueeze 後權重形狀:", weight4.shape)
result4 = weight4 * feature
print("相乘結果形狀:", result4.shape)
print("結果:\n", result4)
print("-" * 50)

# 結果比較
print("所有結果是否相等?")
print("0次和1次相等:", torch.allclose(result0, result1))
print("0次和2次相等:", torch.allclose(result0, result2))
print("0次和3次相等:", torch.allclose(result0, result3))
print("0次和4次相等:", torch.allclose(result0, result4))

原始權重形狀: torch.Size([])
特徵張量形狀: torch.Size([1, 2, 3])
原始權重值: tensor(0.2500)
原始特徵值:
 tensor([[[1, 2, 3],
         [4, 5, 6]]])
--------------------------------------------------
0次 unsqueeze 後權重形狀: torch.Size([])
相乘結果形狀: torch.Size([1, 2, 3])
結果:
 tensor([[[0.2500, 0.5000, 0.7500],
         [1.0000, 1.2500, 1.5000]]])
--------------------------------------------------
1次 unsqueeze 後權重形狀: torch.Size([1])
相乘結果形狀: torch.Size([1, 2, 3])
結果:
 tensor([[[0.2500, 0.5000, 0.7500],
         [1.0000, 1.2500, 1.5000]]])
--------------------------------------------------
2次 unsqueeze 後權重形狀: torch.Size([1, 1])
相乘結果形狀: torch.Size([1, 2, 3])
結果:
 tensor([[[0.2500, 0.5000, 0.7500],
         [1.0000, 1.2500, 1.5000]]])
--------------------------------------------------
3次 unsqueeze 後權重形狀: torch.Size([1, 1, 1])
相乘結果形狀: torch.Size([1, 2, 3])
結果:
 tensor([[[0.2500, 0.5000, 0.7500],
         [1.0000, 1.2500, 1.5000]]])
--------------------------------------------------
4次 unsqueeze 後權重形狀: torch.Size([1, 1, 1,

In [29]:
torch.tensor([0.25])

tensor([0.2500])

In [30]:
'''
        # ===== QKV Attention Pooling =====
        # [batch, seq_len, hidden_size]
        Q = self.q_proj(sequence_output)
        K = self.k_proj(sequence_output)
        V = self.v_proj(sequence_output)
        # Attention scores: [batch, seq_len, seq_len]
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (Q.size(-1) ** 0.5)
        attn_probs = F.softmax(attn_scores, dim=-1)
        # Context vector: [batch, seq_len, hidden_size]
        context_vector_qkv = torch.matmul(attn_probs, V).mean(dim=1)  # [batch, hidden_size]

        # ===== mean pooling =====
        mean_pooled = torch.mean(sequence_output, dim=1)

        # ===== 融合 QKV context 與 mean_pooled =====
        combined_repr = context_vector_qkv + mean_pooled
'''

'\n        # ===== QKV Attention Pooling =====\n        # [batch, seq_len, hidden_size]\n        Q = self.q_proj(sequence_output)\n        K = self.k_proj(sequence_output)\n        V = self.v_proj(sequence_output)\n        # Attention scores: [batch, seq_len, seq_len]\n        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (Q.size(-1) ** 0.5)\n        attn_probs = F.softmax(attn_scores, dim=-1)\n        # Context vector: [batch, seq_len, hidden_size]\n        context_vector_qkv = torch.matmul(attn_probs, V).mean(dim=1)  # [batch, hidden_size]\n\n        # ===== mean pooling =====\n        mean_pooled = torch.mean(sequence_output, dim=1)\n\n        # ===== 融合 QKV context 與 mean_pooled =====\n        combined_repr = context_vector_qkv + mean_pooled\n'

In [32]:
# 1. Generate Q, K, V projections
Q = model.q_proj(sequence_output)  # [batch, seq_len, hidden_size]
K = model.k_proj(sequence_output)  # [batch, seq_len, hidden_size] 
V = model.v_proj(sequence_output)  # [batch, seq_len, hidden_size]

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)


Q shape: torch.Size([1, 3, 896])
K shape: torch.Size([1, 3, 896])
V shape: torch.Size([1, 3, 896])


In [33]:

# 2. Calculate attention scores
attn_scores = torch.matmul(Q, K.transpose(-2, -1))  # [batch, seq_len, seq_len]
print("Attention scores shape:", attn_scores.shape)
print(attn_scores)
attn_scores = attn_scores / (Q.size(-1) ** 0.5)  # Scale by sqrt(hidden_size)
print("\nAttention scores shape:", attn_scores.shape)
print(attn_scores)


Attention scores shape: torch.Size([1, 3, 3])
tensor([[[-1317.0615,   611.3965,   696.0624],
         [  303.6696,   502.5822,   444.3961],
         [  350.6550,   442.7017,   338.4257]]], device='cuda:0',
       grad_fn=<UnsafeViewBackward0>)

Attention scores shape: torch.Size([1, 3, 3])
tensor([[[-43.9999,  20.4253,  23.2538],
         [ 10.1449,  16.7901,  14.8462],
         [ 11.7146,  14.7896,  11.3060]]], device='cuda:0',
       grad_fn=<DivBackward0>)


In [34]:

# 3. Convert to probabilities 
attn_probs = F.softmax(attn_scores, dim=-1)  # [batch, seq_len, seq_len]
print("Attention probabilities shape:", attn_probs.shape)
print(attn_probs)

Attention probabilities shape: torch.Size([1, 3, 3])
tensor([[[5.8497e-30, 5.5804e-02, 9.4420e-01],
         [1.1361e-03, 8.7378e-01, 1.2508e-01],
         [4.2889e-02, 9.2861e-01, 2.8505e-02]]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)


In [35]:

# 4. Calculate context vectors
context_vectors = torch.matmul(attn_probs, V)  # [batch, seq_len, hidden_size]
print("\nContext vectors shape:", context_vectors.shape)
print(context_vectors)


Context vectors shape: torch.Size([1, 3, 896])
tensor([[[ 0.4639,  0.8086, -3.5096,  ..., -0.8570, -0.3313, -0.1655],
         [ 0.9296,  0.6794, -2.3518,  ...,  0.2877, -1.2908,  1.8843],
         [ 1.1300,  0.8665, -1.6050,  ...,  0.6618, -1.4203,  2.3313]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)


In [36]:

# 5. Mean pooling of context vectors
context_vector_qkv = context_vectors.mean(dim=1)  # [batch, hidden_size]
print("Final context vector shape:", context_vector_qkv.shape)
print(context_vector_qkv)

Final context vector shape: torch.Size([1, 896])
tensor([[ 8.4114e-01,  7.8482e-01, -2.4888e+00,  4.1913e-01, -5.8295e-01,
         -1.4376e+00, -4.8050e-01,  1.0628e+00, -3.6586e+00,  7.6769e-01,
         -3.8624e-01, -1.4755e+00,  1.3477e+00, -2.6868e+00, -2.3986e-01,
          9.0790e-01, -1.3222e+00,  3.1644e-01,  9.4292e-01, -2.1251e-02,
          5.8126e-01, -3.1512e+00,  3.4478e+00, -6.4105e-01,  5.0140e+00,
          1.7235e+00, -1.8424e+00,  2.1623e+00,  5.5802e+00,  4.4087e+00,
          2.8230e-01,  2.4496e+00, -2.7407e+00, -3.9626e+00,  3.0564e+00,
          1.5880e+00, -4.0248e-01,  3.6421e+00, -3.1315e+00, -1.3654e+00,
          1.1014e+00,  8.4294e-02,  6.2003e-01,  3.3476e-01, -2.2747e+00,
         -2.3553e-01, -3.0477e+00, -2.4250e+00, -1.5183e+00,  2.4657e+00,
          1.6748e+00, -3.6752e+00, -6.2508e-01, -3.0320e+00,  3.8312e+00,
          9.4705e-01, -1.3498e+00,  5.8513e-01, -1.8945e+00,  7.7629e-02,
         -1.5976e+00, -9.3898e-01,  1.3848e+00, -8.5073e-01,  1

In [38]:

# 6. Mean pooling of original sequence
mean_pooled = torch.mean(sequence_output, dim=1)  # [batch, hidden_size] 
print("\nMean pooled shape:", mean_pooled.shape)
print(mean_pooled)
#print("\nFirst few values of mean pooled:", mean_pooled[0, :5])


Mean pooled shape: torch.Size([1, 896])
tensor([[ 2.3162e+00,  3.1981e-01,  1.0602e+00, -1.2306e+00,  2.7313e-01,
          1.1777e+00, -1.3339e+00, -3.3880e+00, -2.1477e+00,  2.1846e+00,
         -8.2905e-01, -8.1956e-01, -1.3671e+00, -1.2346e+00,  7.0542e-01,
         -1.1324e+00,  7.6071e-01,  2.3275e-01,  8.5672e-01, -2.0268e+00,
         -5.7801e-01,  1.7595e+00, -1.1130e+00,  1.7551e-01,  1.8486e-01,
          2.4680e-01,  9.3545e-01,  9.4052e-01, -2.4416e+00, -1.1927e+00,
         -2.0820e-01,  5.7649e-01, -1.7649e+00, -8.0843e-01,  1.5620e+00,
          3.0185e+00,  1.5160e-01, -1.2221e-01, -9.1823e-01,  1.3643e+00,
         -4.4684e-01, -2.6471e-01,  4.8807e-01, -1.3263e+00, -3.5846e-01,
          4.9100e-01,  1.8304e+00,  4.4776e+00, -8.8017e-01,  3.2318e-01,
          1.0117e+00, -1.6447e+00,  1.9106e+00,  6.7562e+00,  5.2562e-01,
         -1.2864e+00, -8.6189e+00,  1.2444e+00,  2.1778e+01,  3.7381e-01,
          1.8161e-01,  3.6840e-01,  1.5021e+02, -3.1730e-01, -1.5487e-0

In [39]:
# 7. Combine through residual connection
combined_repr = context_vector_qkv + mean_pooled  # [batch, hidden_size]
print("\nCombined representation shape:", combined_repr.shape)
print(combined_repr)



Combined representation shape: torch.Size([1, 896])
tensor([[ 3.1574e+00,  1.1046e+00, -1.4286e+00, -8.1148e-01, -3.0982e-01,
         -2.5989e-01, -1.8144e+00, -2.3252e+00, -5.8062e+00,  2.9523e+00,
         -1.2153e+00, -2.2951e+00, -1.9421e-02, -3.9215e+00,  4.6557e-01,
         -2.2450e-01, -5.6147e-01,  5.4920e-01,  1.7996e+00, -2.0480e+00,
          3.2544e-03, -1.3917e+00,  2.3348e+00, -4.6554e-01,  5.1989e+00,
          1.9703e+00, -9.0691e-01,  3.1028e+00,  3.1385e+00,  3.2160e+00,
          7.4101e-02,  3.0261e+00, -4.5056e+00, -4.7711e+00,  4.6185e+00,
          4.6065e+00, -2.5088e-01,  3.5198e+00, -4.0498e+00, -1.1424e-03,
          6.5458e-01, -1.8042e-01,  1.1081e+00, -9.9154e-01, -2.6332e+00,
          2.5546e-01, -1.2173e+00,  2.0527e+00, -2.3985e+00,  2.7889e+00,
          2.6865e+00, -5.3199e+00,  1.2855e+00,  3.7241e+00,  4.3568e+00,
         -3.3934e-01, -9.9687e+00,  1.8295e+00,  1.9883e+01,  4.5144e-01,
         -1.4160e+00, -5.7058e-01,  1.5160e+02, -1.1680e+00

In [40]:
# Print the first few values to verify
print("\nFirst few values of combined representation:", combined_repr[0, :5])


First few values of combined representation: tensor([ 3.1574,  1.1046, -1.4286, -0.8115, -0.3098], device='cuda:0',
       grad_fn=<SliceBackward0>)


In [41]:
# 5. Classification
print("\n5. Classification:")
logits = model.classifier(combined_repr)
print(f"Final logits shape: {logits.shape}")
logits


5. Classification:
Final logits shape: torch.Size([1, 2])


tensor([[0.7246, 0.4772]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [42]:
# Show probabilities
probs = F.softmax(logits, dim=-1)
for i, prob in enumerate(probs[0]):
    print(f"Class {i}: {prob:.4f}")

Class 0: 0.5615
Class 1: 0.4385


# QwenForClassifier 模型說明與教學

## 一、模型大步驟概念介紹

本模型是基於大型語言模型（如 Qwen2.5-0.5B-instruct）的**情緒分類器**，採用**多層融合**與**QKV注意力池化**，並結合**平均池化**與**殘差連接**，最後經過多層分類器進行分類。

### 整體流程：

1. **特徵提取**：從基礎語言模型取得所有隱藏層（hidden states）。
2. **多層融合**：將最後 N 層（如 4 層）隱藏狀態進行加權融合，獲得更豐富的語義特徵。
3. **QKV 注意力池化**：將融合後的特徵進行 QKV 投影，計算自注意力，獲得序列的語義重點表示。
4. **平均池化**：對融合特徵做平均，保留全局語義。
5. **殘差連接**：將 QKV 注意力池化結果與平均池化結果相加，融合局部與全局特徵。
6. **分類器**：經過多層神經網絡，輸出最終分類結果。

---

## 二、詳細步驟說明與展示

### 1. 特徵提取與多層融合

```python
# 取得所有隱藏層
outputs = self.base_model(
    input_ids=input_ids, 
    attention_mask=attention_mask,
    output_hidden_states=True
)
hidden_states = outputs.hidden_states  # [num_layers, batch, seq_len, hidden_size]

# 取最後 N 層
last_layers = hidden_states[-self.num_fusion_layers:]  # N=4

# 層權重 softmax
layer_weights = F.softmax(self.layer_weights, dim=0)  # [num_fusion_layers]

# 加權融合
sequence_output = torch.zeros_like(last_layers[0])  # [batch, seq_len, hidden_size]
for i, layer in enumerate(last_layers):
    sequence_output += layer_weights[i].unsqueeze(-1).unsqueeze(-1) * layer
```

### 2. QKV 注意力池化

```python
# Q, K, V 線性投影
Q = self.q_proj(sequence_output)  # [batch, seq_len, hidden_size]
K = self.k_proj(sequence_output)
V = self.v_proj(sequence_output)

# 計算注意力分數
attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (Q.size(-1) ** 0.5)  # [batch, seq_len, seq_len]
attn_probs = F.softmax(attn_scores, dim=-1)  # [batch, seq_len, seq_len]

# 加權求 context 向量
context_vector_qkv = torch.matmul(attn_probs, V)  # [batch, seq_len, hidden_size]

# 池化（平均所有 token）
context_vector_qkv = context_vector_qkv.mean(dim=1)  # [batch, hidden_size]
```

### 3. 平均池化

```python
mean_pooled = torch.mean(sequence_output, dim=1)  # [batch, hidden_size]
```

### 4. 殘差連接（融合 QKV 與平均池化）

```python
combined_repr = context_vector_qkv + mean_pooled  # [batch, hidden_size]
```

### 5. 分類器預測

```python
logits = self.classifier(combined_repr)  # [batch, num_labels]
```

### 6. 損失計算（訓練時）

```python
loss = None
if labels is not None:
    loss_fct = nn.CrossEntropyLoss()
    loss = loss_fct(logits, labels)
return {"loss": loss, "logits": logits}
```

---

## 三、圖解與重點

- **多層融合**：讓模型同時利用不同層的語義特徵，提升泛化能力。
- **QKV 注意力池化**：自動學習關注序列中最重要的 token，提升語義聚焦能力。
- **平均池化**：保留全局語義，避免只聚焦於局部。
- **殘差連接**：融合局部重點與全局語義，提升模型穩定性與表達力。
- **多層分類器**：增強非線性映射能力，提升分類準確率。

---

## 四、總結

本模型結合了**多層融合**、**QKV注意力池化**、**平均池化**與**殘差連接**等現代 NLP 技術，能有效提升文本分類的準確率，適合用於教學與實務應用。

# 殘差連接的意義與應用

殘差連接（Residual Connection）是深度學習中的一種重要技術，在這個情緒分類模型中具有關鍵作用。

## 1. 基本概念

殘差連接是指將某一層的輸入直接加到該層的輸出上，形成"捷徑連接"（shortcut connection）。數學表示為：

```
y = F(x) + x
```

其中：
- x 是輸入
- F(x) 是經過某層神經網絡的處理
- y 是最終輸出

## 2. 在情緒分類器中的應用

在這個QwenForClassifier模型中，殘差連接體現在：

```python
# 結合注意力池化和平均池化 (殘差連接)
combined_repr = context_vector + mean_pooled
```

這裡進行了兩種不同特徵表示的相加融合：
- `context_vector`：通過注意力機制獲得的序列表示，突出重要部分
- `mean_pooled`：通過平均池化獲得的序列表示，保留全局信息

## 3. 為什麼要使用殘差連接？

1. **緩解梯度消失/爆炸問題**：
   - 當網絡層數很深時，反向傳播的梯度容易消失
   - 殘差連接提供梯度流動的捷徑，使訊息能更有效傳遞

2. **融合不同類型的特徵表示**：
   - 注意力機制捕獲關鍵詞語和語義重點
   - 平均池化保留完整文本的整體語義
   - 兩者相加可以結合優勢，得到更全面的表示

3. **提高模型表達能力**：
   - 模型可以同時學習局部重要特徵和全局語境特徵
   - 殘差連接使模型能更容易學習恒等映射，保留原始信息

## 4. 與傳統連接方式的區別

- **拼接（Concatenation）**：將特徵向量並排組合，維度增加
- **殘差連接（Addition）**：特徵向量直接相加，維度不變
- **非線性變換後融合**：先經過非線性層再融合

殘差連接相較於拼接更節省參數量，計算效率更高，且能夠保持特徵的原始語義。

在情感分析中，這種融合方式能夠同時關注文本中表達情感的關鍵詞和整體語境，提高分類準確率。